# Scaling: Strong Scale

In [ ]:
!lscpu

In [ ]:
!lscpu --all --extended

In [ ]:
from ray import tune
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

In [ ]:
path = "./ray_results/strong_scaling_
analysis = tune.ExperimentAnalysis(experiment_checkpoint_path=path)

In [ ]:
df = analysis.dataframe()
print(f"{len(df)=}")

In [ ]:
df.keys()

In [ ]:
keys = ["ApplyOverSystems/total", "ApplyOverSystemsSeq/total", "UpdateKinematics/total", "ComputeInternalDynamics/total", "UpdateDynamics/total"]
df["Walltime"] = df[keys].sum(1)
df["num_rods"] = df["config/z_num_rods"]

In [ ]:
desire_field = ["Walltime", "config/block_size", "config/num_rods", "config/num_threads", "n_blocks", "num_rods"]
df = df.filter(items=desire_field).dropna()

## Define Functions

In [ ]:
from scipy.optimize import curve_fit

def amdahl(x, s):
    return 1.0 / (s+((1.0-s)/x))

def get_serial_fraction(speed_up, func, max_core=64):
    """
    Return S: serial speedup
    """
    speed_up = speed_up[speed_up.index <= max_core]
    
    popt, pcov = curve_fit(func, speed_up.index, speed_up)
    #res = minimize_scalar(func)
    return popt[0]

## Strong Scaling

In [ ]:
group_criteria = ["num_rods"]
d = dict(tuple(df.groupby(group_criteria, dropna=True)))

In [ ]:
problem_sizes = d.keys()
problem_sizes

In [ ]:
block_sizes = df["config/block_size"].unique()
block_sizes.sort()
block_sizes

In [ ]:
x_field = "config/num_threads"
y_field = "Walltime"

max_core = 16
elem_per_rod = 16
interpol_x = np.linspace(1, max_core, 100)

for block_size in block_sizes:
    fig = plt.figure(figsize=(12,6))
    ax1 = fig.add_subplot(121)
    ax2 = fig.add_subplot(122)

    best_serial_fraction = 1.0
    for key in d.keys():
        t = d[key].loc[d[key]["config/block_size"]==block_size].groupby(x_field)[y_field]
        t = t.min()
        t = t[t.index <= max_core]
        speed_up = t[1] / t
        speed_up = speed_up.sort_index()
        efficiency = speed_up / speed_up.index
        
        n_elem_log2 = np.log2(key*elem_per_rod)
        ax1.plot(speed_up, label=f"n_elem: 2^{n_elem_log2}")
        ax2.plot(100*efficiency, label=f"n_elem: 2^{n_elem_log2}")

        serial_fraction = get_serial_fraction(speed_up, func=amdahl, max_core=32)
        best_serial_fraction = min(serial_fraction, best_serial_fraction)

    # Configure Speed Up Plot
    best_fit_string = f"best fit S={best_serial_fraction:.02f}"
    
    speed_up_pred = amdahl(interpol_x, best_serial_fraction)
    ax1.plot(interpol_x, speed_up_pred, '-.', label=best_fit_string)
    ax1.plot(np.arange(1,max_core), np.arange(1,max_core), 'r--', label="ideal")

    ax1.legend()
    ax1.set_xlabel('N Threads')
    ax1.set_ylabel("Speed Up")
    ax1.set_title(f"Speed Up : Strong Scale ({block_size=})")

    # Configure Efficiency Plot
    efficiency_pred = speed_up_pred / interpol_x
    ax2.plot(interpol_x, 100*efficiency_pred, '-.', label=best_fit_string)
    ax2.legend()
    ax2.set_xlabel("N Threads")
    ax2.set_ylabel("Efficiency %")
    ax2.set_title(f"Efficiency : Strong Scale ({block_size=})")
    ax2.set_xlim(-1,24)

    print(f"{block_size=}: {best_serial_fraction=}")